In [ ]:
import requests
from rich import print

response = requests.get("https://www.tradingview.com/heatmap/stock/#%7B%22dataSource%22%3A%22SPX500%22%2C%22blockColor%22%3A%22change%22%2C%22blockSize%22%3A%22market_cap_basic%22%2C%22grouping%22%3A%22sector%22%7D")

print(response)

In [ ]:
import os

from dotenv import load_dotenv
from google.genai import Client, types

load_dotenv()


def get_web() -> str:
    """Get the ETF data for a given ETF symbol.

    It uses the Gemini URL context tool to get the data from the ETF Database website.

    Gemini does not support structured output when using the URL context tool, so it is separated into two steps.
    """
    client = Client(
        api_key=os.getenv("GEMINI_API_KEY"),
        http_options={"timeout": 600000},  # 10 minutes timeout
    )

    # Step 1: Get raw data using URL context tool
    url_context_tool = types.Tool(url_context=types.UrlContext())

    raw_response = client.models.generate_content(
        model=os.getenv("FAST_LLM"),
        contents=f"""Ehttps://www.tradingview.com/heatmap/stock/#%7B%22dataSource%22%3A%22SPX500%22%2C%22blockColor%22%3A%22change%22%2C%22blockSize%22%3A%22market_cap_basic%22%2C%22grouping%22%3A%22sector%22%7D What do you see? Can you list some names and numbers you see?""",
        config=types.GenerateContentConfig(
            tools=[url_context_tool],
            temperature=0,
            response_modalities=["TEXT"],
        ),
    )

    return raw_response


get_web()

In [ ]:
response.text

In [ ]:
from rich import print

from stock_search.etf import get_etf_data

print(get_etf_data("VOOG"))

In [ ]:
from rich import print

from stock_search.quote import get_quote

quote = get_quote("AMD")
print(quote)

In [ ]:
from rich import print

from stock_search.quote import batch_get_quote

symbols = ["NVDA", "AMD", "GOOG", "BA", "GE", "RTX", "LMT", "VOOG", "AMZN"]
results = batch_get_quote(symbols)

print(results)

In [1]:
from rich import print

from stock_search.indicators import StockIndicator

indicator = StockIndicator("AMZN")

print(indicator.get_all_indicators())

{
    'analyst_rating': 'Strong Buy',
    'earning_direction': 'Decrease',
    'eps_growth_percent': -6.1068702290076216,
    'expected_earnings_percent': 49.607,
    'fifty_day_change_percent': -1.3123752,
    'ma_strategy': '🚀 STRONG BULLISH - All signals aligned, price well above 200MA',
    'market_cap': '2.290287116288B',
    'reasonable_priced': True,
    'two_hundred_day_change_percent': 2.4730448,
    'upside_downside': 21.0710128055879,
    'volume_surge_percent': 140.61828026205467
}

In [ ]:
import yfinance as yf
from rich import print

ticker = yf.Ticker("AMD")
quote = ticker.info

print(quote)

In [ ]:
quote["earningsGrowth"], quote["grossMargins"]

In [ ]:
(quote["forwardEps"] / quote["trailingEps"] - 1) * 100

In [ ]:
quote["beta"]

In [ ]:
# Volume surge %
(quote["volume"] / quote["averageVolume10days"] - 1) * 100

In [ ]:
# Forward P/E can be lower than the trailing P/E if a company is expected to increase its earnings in the coming year, vice versa
# > 0 when analysts expect earnings to grow faster than price
(quote["trailingPE"] - quote["forwardPE"]) / quote["trailingPE"]

In [ ]:
quote["trailingPE"] / (quote["earningsGrowth"] * 100)

In [ ]:
market_cap = quote["marketCap"] / 1e12 if quote["marketCap"] > 1e12 else quote["marketCap"] / 1e9
market_cap

In [ ]:
quote["fiftyDayAverageChangePercent"] * 100

In [ ]:
quote["twoHundredDayAverageChangePercent"] * 100

In [ ]:
"Bullish" if quote["fiftyDayAverageChangePercent"] > 0 and quote["twoHundredDayAverageChangePercent"] > 0 else ("Mixed" if quote["fiftyDayAverageChangePercent"] * quote["twoHundredDayAverageChangePercent"] < 0 else "Bearish")

In [ ]:
quote["fiftyDayAverageChangePercent"] > quote["twoHundredDayAverageChangePercent"]

In [ ]:
# Upside / Downside
(quote["targetMedianPrice"] / quote["currentPrice"] - 1) * 100

In [ ]:
quote["averageAnalystRating"].split(" - ")[1]

In [ ]:
quote["trailingEps"], quote["forwardEps"], quote["earningsGrowth"]

In [ ]:
(quote["forwardEps"] / quote["trailingEps"] - 1)

In [ ]:
quote["grossMargins"]

In [ ]:
(quote["forwardEps"] / quote["trailingEps"] - 1) * quote["grossMargins"]

In [ ]:
from stock_search.utils import datetime_to_str

datetime_to_str(quote["earningsTimestamp"])

In [ ]:
import requests
from rich import print

response = requests.get("https://financialmodelingprep.com/stable/quote?symbol=NVDA&apikey=qHXRYd847oSWxztRTpzGp5XLB5QjbH9B").json()

print(response)

In [ ]:
import yfinance as yf
from rich import print

ticker = yf.Ticker("NVDA")
quote = ticker.info

print(quote)

In [ ]:
from rich import print

from stock_search.news import get_news_yfinance

articles = get_news_yfinance("AMD")

len(articles)
print(articles)